# Batch Evaluation and Metrics

Simulate batch results offline, then optionally run live batch evaluation.

In [ ]:
import sys
from pathlib import Path

repo_src = Path("..").resolve() / "src"
if repo_src.exists():
    sys.path.insert(0, str(repo_src))

import pandas as pd
from openjury.output_format import AgentEvalResult, CriterionEvaluation, TrialResult
from openjury.scoring import JurorScore, ScoredMetrics

In [ ]:
# Simulate batch results (offline)
def make_result(case_id: str, score: float, agreement: float) -> dict:
    metrics = ScoredMetrics(
        weighted_mean=score, mean=score, median=score,
        min_score=score - 0.3, max_score=score + 0.3,
        harmonic_mean=score - 0.1, weakest_link=0.5,
        juror_agreement=agreement,
    )
    return {
        "case_id": case_id,
        "composite_score": score,
        "juror_agreement": agreement,
        "normalized": score / 5,
    }

rows = [
    make_result("case-1", 4.2, 0.91),
    make_result("case-2", 3.1, 0.72),
    make_result("case-3", 4.8, 0.95),
    make_result("case-4", 2.9, 0.65),
]
df = pd.DataFrame(rows)
df

In [ ]:
# Summary statistics
print("Mean composite:", df["composite_score"].mean().round(2))
print("Min composite:", df["composite_score"].min())
print("Low agreement cases:", df[df["juror_agreement"] < 0.8]["case_id"].tolist())

In [ ]:
# Agreement heatmap-style view
pivot = df.set_index("case_id")[["composite_score", "juror_agreement"]]
pivot.style.background_gradient(cmap="RdYlGn", subset=["composite_score", "juror_agreement"])

In [ ]:
# Optional: live batch with mock agent + OPENAI_API_KEY
import os
from openjury import EvaluationItem, ExecutionOptions, JuryConfig, OpenJury
from openjury.endpoint_fetcher import load_endpoints_file

if os.environ.get("OPENAI_API_KEY"):
    cfg = JuryConfig.from_json_file("../examples/basic_usage/config.json")
    endpoint = load_endpoints_file("../examples/basic_usage/endpoints.json")[0]
    jury = OpenJury(cfg)
    items = [
        EvaluationItem(prompt="How do I reset my password?", item_id="nb-1"),
        EvaluationItem(prompt="How do I cancel my subscription?", item_id="nb-2"),
    ]
    results = jury.evaluate_items(items, endpoint, options=ExecutionOptions(max_item_workers=2))
    live_rows = [
        {
            "case_id": r.item.item_id,
            "composite_score": r.result.composite_score if r.result else None,
            "error": str(r.error) if r.error else None,
        }
        for r in results
    ]
    pd.DataFrame(live_rows)
else:
    print("Set OPENAI_API_KEY and start mock agent on :8080 for live batch eval.")